In [1]:
import torch

In [2]:
# add aura to the path temporarily which is outside this directory
import sys
sys.path.append('../')

In [3]:
import aura

c:\Users\adity\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
model = aura.models.LightningATCNet(
    num_classes=2,
    n_chans=19,
    n_outputs=2,
)

In [9]:
import torchmetrics
import torch

In [8]:
accu = torchmetrics.Accuracy(task="multiclass", num_classes=2)

In [13]:
ytrue = torch.tensor(
    [
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
    ]
)
ytrue = torch.argmax(ytrue, dim=1)
ypred = torch.tensor(
    [
        [0.9, 0.1],
        [0.1, 0.9],
        [0.9, 0.1],
        [0.1, 0.9],
    ]
)
ypred = torch.argmax(ypred, dim=1)
print(accu(ypred, ytrue))

tensor(1.)


In [18]:
from lightning import LightningDataModule
import numpy as np

In [41]:
class SSLDataModule(LightningDataModule):
    """
    Subject Specific LightningDataModule
    """
    class _Dataset(torch.utils.data.Dataset):
        def __init__(self, X, Y, n_outputs=2):
            self.X = X
            self.Y = Y
            self.n_outputs = n_outputs

        def __len__(self):
            return len(self.X)

        def __getitem__(self, idx):
            x = torch.tensor(self.X[idx], dtype=torch.float32)
            y = torch.tensor(self.Y[idx], dtype=torch.long)
            y = torch.nn.functional.one_hot(y, num_classes=self.n_outputs)

    def __init__(
        self,
        X, 
        Y,
        S,
        test_size=0.1,
        val_size=0.1,
        batch_size=32,
        num_workers=0,
        pin_memory=False,
        shuffle=True,
    ):
        super().__init__()
        self.X = X
        self.Y = Y
        self.S = S
        self.test_size = test_size
        self.val_size = val_size
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.pin_memory = pin_memory
        self.shuffle = shuffle

    def setup(self, stage=None):
        unique_subjects = np.unique(self.S)
        ns = len(unique_subjects)
        nte = int(self.test_size * ns)
        nva = int(self.val_size * ns)
        np.random.shuffle(unique_subjects)
        tes = unique_subjects[:nte]
        vas = unique_subjects[nte:nte+nva]
        trs = unique_subjects[nte+nva:]
        self.trs = trs
        self.vas = vas
        self.tes = tes

    def train_dataloader(self):
        idx = np.isin(self.S, self.trs)
        X = self.X[idx]
        Y = self.Y[idx]
        return torch.utils.data.DataLoader(
            self._Dataset(X, Y),
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
            shuffle=self.shuffle,
        )
    
    def val_dataloader(self):
        idx = np.isin(self.S, self.vas)
        X = self.X[idx]
        Y = self.Y[idx]
        return torch.utils.data.DataLoader(
            self._Dataset(X, Y),
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
            shuffle=self.shuffle,
        )
    
    def test_dataloader(self):
        idx = np.isin(self.S, self.tes)
        X = self.X[idx]
        Y = self.Y[idx]
        return torch.utils.data.DataLoader(
            self._Dataset(X, Y),
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
            shuffle=self.shuffle,
        )

In [30]:
X.shape

(100, 19, 1000)

In [ ]:
torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X, Y),
    batch_size=16,
    num_workers=0,
)

In [42]:
X = np.random.rand(100, 19, 1000)
Y = np.random.randint(0, 2, 100)
S = np.random.randint(0, 10, 100)

dm = SSLDataModule(X, Y, S)
dm.setup()

for i in dm.train_dataloader():
    print(i[1])
    break

tensor([0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0,
        1, 1, 0, 0, 0, 0, 0, 1], dtype=torch.int32)


In [33]:
torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X, Y),
    batch_size=16,
    num_workers=0,
)

TypeError: 'int' object is not callable

In [34]:
torch.utils.data.TensorDataset(X, Y)

TypeError: 'int' object is not callable